# `build_manager_matches` — Build the Manager Match Dataset

## Purpose

Extract match-level data for eight managers from the local Transfermarkt database (`tm_football.db`) and produce a clean, flat CSV (`manager_matches.csv`) ready for statistical analysis.

This notebook is the bridge between the data lake (built by `tm_scraper`) and the analysis notebooks. It runs in under a minute and can be re-run any time the database is updated.

## Input

`tm_football.db` — local SQLite database with two tables:
- `game_index`: one row per match, basic metadata and score
- `matches`: same, enriched with manager names, formations, attendance

## Output

`data/processed/manager_matches.csv` — one row per manager per match, with:
- match metadata: date, league, season, round, clubs
- goals for and against (from the manager's perspective)
- derived columns: goal difference, result (W/D/L), home/away flag

## Managers in scope

Massimiliano Allegri, Pep Guardiola, José Mourinho, Antonio Conte, Carlo Ancelotti, Roberto De Zerbi, Vincenzo Italiano, Simone Inzaghi.

Big 5 domestic leagues only: Premier League, Serie A, La Liga, Bundesliga, Ligue 1.

## Notebook structure

| Block | Content |
|-------|---------|
| A | Setup: imports, paths, constants, schema inspection |
| B | Extract and transform: query, unpivot, parse dates, derive columns |
| C | Save and verify |

## Block A — Setup

Imports, paths, and constants. Run this block first in every session. The schema inspection cells (A.3, A.4, A.5) are diagnostic. Run once to verify the database, skip on subsequent runs.

In [1]:
# ── A.1 Imports ───────────────────────────────────────────────────────────────
import sqlite3
import pandas as pd
from pathlib import Path

print("OK")

OK


In [2]:
# ── A.2 Paths and constants ───────────────────────────────────────────────────
DB_PATH = Path(r"C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\tm_football.db")
OUT_PATH = DB_PATH.parent / "./data/processed/manager_matches.csv"

BIG5 = {"GB1", "IT1", "ES1", "L1", "FR1"}

MANAGERS = [
    "Massimiliano Allegri",
    "Pep Guardiola",
    "José Mourinho",
    "Antonio Conte",
    "Carlo Ancelotti",
    "Roberto De Zerbi",
    "Vincenzo Italiano",
    "Simone Inzaghi",
    "Maurizio Sarri",        # added
]

print("DB path:", DB_PATH)
print("DB exists:", DB_PATH.exists())
print("Output path:", OUT_PATH)
print("Managers:", len(MANAGERS))

DB path: C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\tm_football.db
DB exists: True
Output path: C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\data\processed\manager_matches.csv
Managers: 9


## Inspecting the DB schema before writing any query.

In [3]:
# ── A.3 Schema inspection ─────────────────────────────────────────────────────
con = sqlite3.connect(DB_PATH)

for table in ["game_index", "matches"]:
    print(f"\n── {table} ──")
    cur = con.execute(f"PRAGMA table_info({table})")
    for row in cur.fetchall():
        print(f"  {row[1]:30s}  {row[2]}")


── game_index ──
  game_id                         TEXT
  league_code                     TEXT
  season                          INTEGER
  round                           TEXT
  date                            TEXT
  home_club                       TEXT
  away_club                       TEXT
  home_goals                      INTEGER
  away_goals                      INTEGER
  indexed_at                      TEXT

── matches ──
  game_id                         TEXT
  league_code                     TEXT
  season                          INTEGER
  round                           TEXT
  date                            TEXT
  home_club                       TEXT
  away_club                       TEXT
  home_goals                      INTEGER
  away_goals                      INTEGER
  home_manager                    TEXT
  away_manager                    TEXT
  home_formation                  TEXT
  away_formation                  TEXT
  attendance                      INTEGER
  referee 

## row count and null check

In [4]:
# ── A.4 Row counts and manager null check ─────────────────────────────────────
cur = con.execute("SELECT COUNT(*) FROM matches")
total = cur.fetchone()[0]

cur = con.execute("SELECT COUNT(*) FROM matches WHERE home_manager IS NULL")
null_home = cur.fetchone()[0]

cur = con.execute("SELECT COUNT(*) FROM matches WHERE away_manager IS NULL")
null_away = cur.fetchone()[0]

print(f"Total rows in matches : {total:>8,}")
print(f"NULL home_manager     : {null_home:>8,}  ({null_home/total*100:.1f}%)")
print(f"NULL away_manager     : {null_away:>8,}  ({null_away/total*100:.1f}%)")

Total rows in matches :   42,898
NULL home_manager     :      651  (1.5%)
NULL away_manager     :      721  (1.7%)


##  Check how our managers actually appear in the data.

In [5]:
# ── A.5 Manager name check ────────────────────────────────────────────────────
for name in MANAGERS:
    cur = con.execute("""
        SELECT COUNT(*) FROM matches
        WHERE home_manager = ? OR away_manager = ?
    """, (name, name))
    n = cur.fetchone()[0]
    print(f"{n:>5}  {name}")

  544  Massimiliano Allegri
  633  Pep Guardiola
  591  José Mourinho
  393  Antonio Conte
  531  Carlo Ancelotti
  287  Roberto De Zerbi
  228  Vincenzo Italiano
  349  Simone Inzaghi
  370  Maurizio Sarri


## Block B — Extract and Transform

Query `matches` for our eight managers, unpivot from wide (one row per match) to long (one row per manager per match), parse the date column — which comes in three different formats from the scraper — and add derived columns.

### B.1 Query the data: one row per manager per match, unpivoted. 

In [6]:
# ── B.1 Query: one row per manager per match ──────────────────────────────────
query = """
SELECT
    game_id,
    date,
    league_code,
    season,
    round,
    home_manager   AS manager,
    away_manager   AS opponent_manager,
    home_club      AS manager_club,
    away_club      AS opponent_club,
    home_goals     AS goals_for,
    away_goals     AS goals_against,
    1              AS is_home
FROM matches
WHERE home_manager IN ({placeholders})
  AND league_code IN ({big5})

UNION ALL

SELECT
    game_id,
    date,
    league_code,
    season,
    round,
    away_manager   AS manager,
    home_manager   AS opponent_manager,
    away_club      AS manager_club,
    home_club      AS opponent_club,
    away_goals     AS goals_for,
    home_goals     AS goals_against,
    0              AS is_home
FROM matches
WHERE away_manager IN ({placeholders})
  AND league_code IN ({big5})
""".format(
    placeholders=",".join("?" * len(MANAGERS)),
    big5=",".join("?" * len(BIG5)),
)

params = MANAGERS + list(BIG5) + MANAGERS + list(BIG5)
df = pd.read_sql_query(query, con, params=params)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Rows: 3,858
Columns: ['game_id', 'date', 'league_code', 'season', 'round', 'manager', 'opponent_manager', 'manager_club', 'opponent_club', 'goals_for', 'goals_against', 'is_home']


3,488 rows, roughly double the per-manager counts we saw in A.5 (each match appears once as home, once as away when two of our managers faced each other, otherwise once). Looks right.

### B.2. Derived columns

In [7]:
# ── B.2 Derived columns ───────────────────────────────────────────────────────
df["goal_diff"] = df["goals_for"] - df["goals_against"]

df["result"] = df["goal_diff"].apply(
    lambda gd: "W" if gd > 0 else ("L" if gd < 0 else "D")
)

df["date"] = pd.to_datetime(df["date"])

print(df[["manager", "date", "goals_for", "goals_against", "goal_diff", "result"]].head(10))

C:\Users\pacor\AppData\Local\Temp\ipykernel_20484\1483743160.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"])


DateParseError: Unknown datetime string format, unable to parse: 2. Matchday|Sat, 13/09/08|  8:00 PM

In [8]:
# ── B.2 diagnostic: check date column sample ─────────────────────────────────
print(df["date"].dtype)
print()
print(df["date"].unique()[:20])

str

<StringArray>
[  '2. Matchday|Sat, 13/09/08|  8:00 PM',
   '4. Matchday|Wed, 24/09/08|  8:00 PM',
  '6. Matchday|Sat, 04/10/08|  10:00 PM',
  '8. Matchday|Sat, 25/10/08|  10:00 PM',
 '10. Matchday|Sat, 08/11/08|  11:00 PM',
  '12. Matchday|Sun, 23/11/08|  7:00 PM',
 '14. Matchday|Sat, 06/12/08|  10:00 PM',
 '15. Matchday|Sat, 13/12/08|  10:00 PM',
  '17. Matchday|Sat, 03/01/09|  8:00 PM',
  '19. Matchday|Sat, 17/01/09|  8:00 PM',
 '20. Matchday|Sat, 24/01/09|  10:00 PM',
  '22. Matchday|Sun, 08/02/09|  7:00 PM',
  '24. Matchday|Sat, 21/02/09|  8:00 PM',
  '26. Matchday|Sat, 07/03/09|  8:00 PM',
  '28. Matchday|Sun, 22/03/09|  7:00 PM',
  '30. Matchday|Sat, 11/04/09|  8:00 PM',
  '35. Matchday|Sun, 10/05/09|  7:00 PM',
  '37. Matchday|Sat, 23/05/09|  9:00 PM',
  '1. Matchday|Mon, 31/08/09|  10:00 PM',
  '3. Matchday|Sat, 19/09/09|  10:00 PM']
Length: 20, dtype: str


The date is buried in the middle field of a pipe-separated string. Easy to extract.

In [9]:
# ── B.2 fix: extract date from pipe-separated string ─────────────────────────
def parse_tm_date(s):
    # format: "N. Matchday|Day, DD/MM/YY|  HH:MM PM"
    try:
        part = s.split("|")[1].strip()        # "Sat, 13/09/08"
        part = part.split(", ")[1].strip()    # "13/09/08"
        return pd.to_datetime(part, format="%d/%m/%y")
    except Exception:
        return pd.NaT

df["date"] = df["date"].apply(parse_tm_date)

print(f"NaT count: {df['date'].isna().sum()}")
print(df["date"].head(5))

NaT count: 2539
0   2008-09-13
1   2008-09-24
2   2008-10-04
3   2008-10-25
4   2008-11-08
Name: date, dtype: datetime64[us]


In [10]:
# ── B.2 diagnostic: what do the NaT rows look like? ──────────────────────────
mask = df["date"].isna()
print(df.loc[mask, "date"].head(20))  # this will be NaT, so check the raw values

# re-query just the raw date strings for NaT rows
raw_dates = pd.read_sql_query(query, con, params=params)["date"]
print(raw_dates[mask.values].unique()[:20])

31    NaT
39    NaT
113   NaT
114   NaT
115   NaT
116   NaT
117   NaT
118   NaT
119   NaT
120   NaT
121   NaT
122   NaT
123   NaT
124   NaT
125   NaT
126   NaT
127   NaT
128   NaT
129   NaT
130   NaT
Name: date, dtype: datetime64[us]
<StringArray>
['28. Matchday24/03/108:00 PM',  '4. Matchday22/09/108:00 PM',
                  '2012-09-30',                  '2012-10-20',
                  '2012-11-03',                  '2012-09-02',
                  '2013-05-04',                  '2013-04-20',
                  '2013-05-08',                  '2012-11-17',
                  '2013-06-01',                  '2012-12-01',
                  '2013-01-06',                  '2012-12-16',
                  '2013-01-27',                  '2013-02-09',
                  '2013-02-17',                  '2013-03-02',
                  '2013-03-16',                  '2013-04-06']
Length: 20, dtype: str


Three formats in the wild: the pipe format, a no-pipe variant, and plain ISO YYYY-MM-DD. Let's handle all three.

In [11]:
# ── B.2 fix: handle all three date formats ────────────────────────────────────
def parse_tm_date(s):
    if not isinstance(s, str):
        return pd.NaT
    # Format 1: "N. Matchday|Day, DD/MM/YY|  HH:MM PM"
    if "|" in s:
        try:
            part = s.split("|")[1].strip()
            part = part.split(", ")[1].strip()
            return pd.to_datetime(part, format="%d/%m/%y")
        except Exception:
            return pd.NaT
    # Format 2: "N. MatchdayDD/MM/YYHH:MM PM" (no pipe, no comma)
    if "Matchday" in s:
        try:
            part = s.split("Matchday")[1].strip()[:8]
            return pd.to_datetime(part, format="%d/%m/%y")
        except Exception:
            return pd.NaT
    # Format 3: plain ISO "YYYY-MM-DD"
    try:
        return pd.to_datetime(s, format="%Y-%m-%d")
    except Exception:
        return pd.NaT

df["date"] = raw_dates.apply(parse_tm_date)

print(f"NaT count: {df['date'].isna().sum()}")
print(df["date"].describe())

NaT count: 0
count                          3858
mean     2018-08-15 06:02:25.567651
min             2008-08-30 00:00:00
25%             2014-11-12 06:00:00
50%             2019-02-16 00:00:00
75%             2022-10-16 00:00:00
max             2026-05-24 00:00:00
Name: date, dtype: object


### B.3 Sanity check

In [13]:
# ── B.3 Sanity check: results per manager ─────────────────────────────────────
summary = (
    df.groupby(["manager", "result"])
    .size()
    .unstack(fill_value=0)
    [["W", "D", "L"]]
)
summary["total"] = summary.sum(axis=1)
summary["win_pct"] = (summary["W"] / summary["total"] * 100).round(1)
print(summary.sort_values("win_pct", ascending=False))

result                  W    D    L  total  win_pct
manager                                            
Pep Guardiola         466   94   73    633     73.6
Antonio Conte         257   78   58    393     65.4
Carlo Ancelotti       340  100   91    531     64.0
Simone Inzaghi        209   63   77    349     59.9
Massimiliano Allegri  322  116  106    544     59.2
José Mourinho         308  114  101    523     58.9
Maurizio Sarri        200   88   82    370     54.1
Vincenzo Italiano      92   59   77    228     40.4
Roberto De Zerbi      108   70  109    287     37.6


That's roughly what we'd expect given:

- Guardiola started in a top-five league in 2008 and has managed continuously.
- Allegri entered Serie A in 2008 and has almost never left it.
- Conte lost league matches while managing the Italian national team.
- Inzaghi started later.
- De Zerbi and Italiano entered the top five leagues much later.

## Block C — Save and Verify

Write the final dataframe to CSV and print a summary to confirm row counts and column names before closing the notebook.

In [14]:
# ── C.1 Save to CSV ───────────────────────────────────────────────────────────
df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Rows : {len(df):,}")
print(f"Cols : {list(df.columns)}")

Saved: C:\Users\pacor\Documents\Notebooks\Python\Inpat Expat\inpat-expat\corto-muso\data\processed\manager_matches.csv
Rows : 3,858
Cols : ['game_id', 'date', 'league_code', 'season', 'round', 'manager', 'opponent_manager', 'manager_club', 'opponent_club', 'goals_for', 'goals_against', 'is_home', 'goal_diff', 'result']


In [17]:
import numpy as np
np.sort(df['manager'].unique())

array(['Antonio Conte', 'Carlo Ancelotti', 'José Mourinho',
       'Massimiliano Allegri', 'Maurizio Sarri', 'Pep Guardiola',
       'Roberto De Zerbi', 'Simone Inzaghi', 'Vincenzo Italiano'],
      dtype=object)

In [ ]:
con.execute("SELECT DISTINCT home_manager FROM matches WHERE home_manager LIKE '%Sarri%'").fetchall()

In [18]:
df.head()

,game_id,date,league_code,season,round,manager,opponent_manager,manager_club,opponent_club,goals_for,goals_against,is_home,goal_diff,result
0,920353,2008-09-13,ES1,2008,NaN,Pep Guardiola,Juan Muñiz,FC Barcelona,Racing Santander,1,1,1,0,D
1,920372,2008-09-24,ES1,2008,NaN,Pep Guardiola,Francisco Chaparro,FC Barcelona,Real Betis Balompié,3,2,1,1,W
2,920391,2008-10-04,ES1,2008,NaN,Pep Guardiola,Javier Aguirre,FC Barcelona,Atlético de Madrid,6,1,1,5,W
3,920410,2008-10-25,ES1,2008,NaN,Pep Guardiola,Gonzalo Arconada,FC Barcelona,UD Almería,5,0,1,5,W
4,920429,2008-11-08,ES1,2008,NaN,Pep Guardiola,José Luis Mendilibar,FC Barcelona,Real Valladolid CF,6,0,1,6,W
